## Learning Matplotlib and Seaborn - Dataviz Libraries

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
DATA = Path("../data")
OUTPUT = Path("../output")
OUTPUT.mkdir(exist_ok=True)

accounts = pd.read_csv(DATA / "accounts.csv")
subscriptions = pd.read_csv(DATA / "subscriptions.csv")
usage = pd.read_csv(DATA / "monthly_usage.csv")

df = subscriptions.merge(accounts, on="account_id")
merged_usage = usage.merge(
    df[["subscription_id", "segment"]], on="subscription_id")
closed = df[df["renewal_status"] != "Open"]

In [ ]:
SEGMENT_COLORS = {
    "Enterprise": "#2a78d6",
    "Mid-Market": "#eb6834",
    "Public Sector": "#1baf7a",
    "SMB": "#eda100",
}

segment_trend = (
    merged_usage.groupby(["month", "segment"])["seat_utilization"]
    .mean()
    .unstack("segment")
)

fig, ax = plt.subplots(figsize=(10, 5))

for segment in segment_trend.columns:
    ax.plot(
        segment_trend.index,
        segment_trend[segment],
        label=segment,
        color=SEGMENT_COLORS[segment],
        linewidth=2,
        marker="o",
        markersize=4,
    )

ax.set_title("Utilization trend by segment")
ax.set_xlabel("Month")
ax.set_ylabel("Average seat utilization")
ax.yaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.tick_params(axis="x", rotation=45)
ax.grid(axis="y", alpha=0.3)
ax.spines[["top", "right"]].set_visible(False)
ax.legend(title="Segment")

fig.tight_layout()
plt.show()

In [ ]:
region_acv = (
    df.groupby("region")["annual_contract_value"]
    .sum()
    .div(1_000_000)
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(8, 5))

bars = ax.bar(region_acv.index, region_acv.values, color="#2a78d6", width=0.6)
ax.bar_label(bars, fmt="${:,.1f}M", padding=3)

ax.set_title("Total ACV by region")
ax.set_xlabel("Region")
ax.set_ylabel("Total ACV ($M)")
ax.margins(y=0.1)
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.spines[["top", "right"]].set_visible(False)

fig.tight_layout()
plt.show()

In [ ]:
TIER_COLORS = {
    "Essentials": "#2a78d6",
    "Advantage": "#eb6834",
    "Premier": "#1baf7a",
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5.5))

for ax, scale in zip(axes, ["linear", "log"]):
    for tier, color in TIER_COLORS.items():
        tier_rows = df[df["tier"] == tier]
        ax.scatter(
            tier_rows["seats_purchased"],
            tier_rows["annual_contract_value"],
            label=tier,
            color=color,
            s=18,
            alpha=0.6,
            edgecolors="white",
            linewidths=0.4,
        )

    ax.set_xscale(scale)
    ax.set_yscale(scale)
    ax.set_title(f"Seats vs ACV by tier : {scale} axes")
    ax.set_xlabel("Seats purchased")
    ax.set_ylabel("Annual contract value")
    ax.xaxis.set_major_formatter(lambda value, _: f"{value:,.0f}")
    ax.yaxis.set_major_formatter(lambda value, _: f"${value:,.0f}")
    ax.grid(alpha=0.3)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)
    ax.legend(title="Tier", loc="upper left")

fig.tight_layout()
plt.show()

In [ ]:
STATUS_COLORS = {
    "Renewed": "#2a78d6",
    "Churned": "#eb6834",
}

fig, ax = plt.subplots(figsize=(10, 5))

sns.histplot(
    data=closed,
    x="avg_seat_utilization",
    hue="renewal_status",
    hue_order=list(STATUS_COLORS),
    palette=STATUS_COLORS,
    bins=30,
    alpha=0.5,
    edgecolor="white",
    ax=ax,
)

ax.set_title("Average seat utilization of closed renewals")
ax.set_xlabel("Average seat utilization")
ax.set_ylabel("Subscriptions")
ax.xaxis.set_major_formatter(lambda value, _: f"{value:.0%}")
ax.get_legend().set_title("Renewal status")
sns.despine(ax=ax)

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

sns.boxplot(
    data=df,
    x="segment",
    y="annual_contract_value",
    hue="segment",
    order=list(SEGMENT_COLORS),
    palette=SEGMENT_COLORS,
    log_scale=True,
    legend=False,
    width=0.5,
    ax=ax,
)

ax.set_title("ACV by segment : log scale")
ax.set_xlabel("Segment")
ax.set_ylabel("Annual contract value")
ax.yaxis.set_major_formatter(lambda value, _: f"${value:,.0f}")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
sns.despine(ax=ax)

fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))

sns.countplot(
    data=closed,
    x="tier",
    hue="renewal_status",
    order=list(TIER_COLORS),
    hue_order=list(STATUS_COLORS),
    palette=STATUS_COLORS,
    width=0.6,
    ax=ax,
)

ax.set_title("Renewal outcome by tier")
ax.set_xlabel("Tier")
ax.set_ylabel("Subscriptions")
ax.grid(axis="y", alpha=0.3)
ax.set_axisbelow(True)
ax.get_legend().set_title("Renewal status")
sns.despine(ax=ax)

fig.tight_layout()
plt.show()

In [ ]:
HEALTH_METRICS = {
    "avg_seat_utilization": "Seat utilization",
    "utilization_trend": "Utilization trend",
    "avg_feature_adoption": "Feature adoption",
    "support_cases_12m": "Support cases (12m)",
    "sev1_cases_12m": "Sev1 cases (12m)",
    "nps_score": "NPS score",
}

correlations = df[list(HEALTH_METRICS)].rename(columns=HEALTH_METRICS).corr()

fig, ax = plt.subplots(figsize=(8, 6.5))

sns.heatmap(
    correlations,
    annot=True,
    fmt=".2f",
    vmin=-1,
    vmax=1,
    cmap="vlag",
    square=True,
    linewidths=1,
    cbar_kws={"label": "Pearson correlation"},
    ax=ax,
)

ax.set_title("Correlation of health metrics")
ax.tick_params(axis="x", rotation=45)
plt.setp(ax.get_xticklabels(), ha="right")

fig.tight_layout()
plt.show()